
# Advanced Bitcoin Fraud Detection using Spark + GNN + CatBoost

This notebook implements:

- Apache Spark preprocessing
- Random Forest baseline
- CatBoost baseline
- Graph Attention Network (GAT)
- Focal Loss for imbalance handling
- Hybrid GAT + CatBoost architecture
- Automatic model saving
- ZIP export for all trained models

Dataset:
- Elliptic Bitcoin Dataset


In [1]:

# Install dependencies

!pip install pyspark torch-geometric catboost -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.5 MB/s eta 0:00:00


In [2]:

import os
import zipfile
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier

from catboost import CatBoostClassifier

from torch_geometric.data import Data
from torch_geometric.nn import GATConv



# Start Spark Session


In [3]:

spark = SparkSession.builder.appName("EllipticFraud").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 11:30:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



# Load Dataset


In [4]:

features_path = "/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_features.csv"
classes_path = "/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_classes.csv"
edges_path = "/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"

features_df = spark.read.csv(features_path, inferSchema=True, header=False)
classes_df = spark.read.csv(classes_path, inferSchema=True, header=True)
edges_df = spark.read.csv(edges_path, inferSchema=True, header=True)

print("Features:", features_df.count())
print("Classes:", classes_df.count())
print("Edges:", edges_df.count())


Features: 203769
Classes: 203769
Edges: 234355



# Preprocessing using Spark


In [5]:

feature_columns = ["txId", "time_step"] + [
    f"f_{i}" for i in range(len(features_df.columns)-2)
]

features_df = features_df.toDF(*feature_columns)

merged_df = features_df.join(classes_df, on="txId", how="inner")

merged_df = merged_df.filter(col("class") != "unknown")

merged_df = merged_df.withColumn(
    "class",
    when(col("class") == "1", 1).otherwise(0)
)

pandas_df = merged_df.toPandas()

print(pandas_df.head())


26/05/17 11:30:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


        txId  time_step       f_0       f_1       f_2        f_3       f_4  \
0  232438397          1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
1  232029206          1 -0.005027  0.578941 -0.091383   4.380281 -0.063725   
2  232344069          1 -0.147852 -0.184668 -1.201369  -0.121970 -0.043875   
3   27553029          1 -0.151357 -0.184668 -1.201369  -0.121970 -0.043875   
4    3881097          1 -0.172306 -0.184668 -1.201369   0.028105 -0.043875   

        f_5        f_6       f_7  ...     f_156     f_157     f_158     f_159  \
0  9.782742  12.414558 -0.163645  ... -0.613614  0.241128  0.241406  1.072793   
1  4.667146   0.851305 -0.163645  ... -0.613614  0.241128  0.241406  0.604120   
2 -0.113002  -0.061584 -0.137933  ... -0.613614  0.241128  0.241406  0.018279   
3 -0.113002  -0.061584 -0.141519  ... -0.582077 -0.979074 -0.978556  0.018279   
4 -0.029140   0.242712 -0.163640  ... -0.600999  0.241128  0.241406  0.018279   

      f_160     f_161     f_162     f_163   


# Build Graph Structure


In [6]:

node_ids = pandas_df["txId"].unique()

id_map = {node_id: idx for idx, node_id in enumerate(node_ids)}

pandas_df["node_idx"] = pandas_df["txId"].map(id_map)

feature_cols = [c for c in pandas_df.columns if c.startswith("f_")]

X = torch.tensor(
    pandas_df[feature_cols].values,
    dtype=torch.float
)

y = torch.tensor(
    pandas_df["class"].astype(int).values,
    dtype=torch.long
)

edges_pd = edges_df.toPandas()

edges_pd = edges_pd[
    edges_pd["txId1"].isin(id_map.keys()) &
    edges_pd["txId2"].isin(id_map.keys())
]

source = edges_pd["txId1"].map(id_map).values
target = edges_pd["txId2"].map(id_map).values

edge_index = torch.tensor(
    [source, target],
    dtype=torch.long
)

data = Data(
    x=X,
    edge_index=edge_index,
    y=y
)

print(data)


Data(x=[46564, 165], edge_index=[2, 36624], y=[46564])


/tmp/ipykernel_57/3098328989.py:29: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor(



# Train/Test Split


In [7]:

indices = np.arange(data.num_nodes)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=y.numpy(),
    random_state=42
)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.test_mask = test_mask



# Random Forest Baseline


In [8]:

X_np = X.numpy()
y_np = y.numpy()

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_np[train_idx], y_np[train_idx])

rf_preds = rf_model.predict(X_np[test_idx])

print("RF Accuracy:", accuracy_score(y_np[test_idx], rf_preds))
print("RF F1:", f1_score(y_np[test_idx], rf_preds))

print(classification_report(y_np[test_idx], rf_preds))


RF Accuracy: 0.9876516697090089
RF F1: 0.9325513196480938
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      8404
           1       1.00      0.87      0.93       909

    accuracy                           0.99      9313
   macro avg       0.99      0.94      0.96      9313
weighted avg       0.99      0.99      0.99      9313




# CatBoost Baseline


In [9]:

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='F1',
    class_weights=[1,5],
    verbose=50
)

cat_model.fit(
    X_np[train_idx],
    y_np[train_idx]
)

cat_preds = cat_model.predict(X_np[test_idx])

print("CatBoost Accuracy:", accuracy_score(y_np[test_idx], cat_preds))
print("CatBoost F1:", f1_score(y_np[test_idx], cat_preds))

print(classification_report(y_np[test_idx], cat_preds))


0:	learn: 0.8990322	total: 176ms	remaining: 52.7s
50:	learn: 0.9610551	total: 4.42s	remaining: 21.6s
100:	learn: 0.9803813	total: 8.63s	remaining: 17s
150:	learn: 0.9919837	total: 12.8s	remaining: 12.7s
200:	learn: 0.9968685	total: 17.1s	remaining: 8.4s
250:	learn: 0.9982698	total: 21.2s	remaining: 4.14s
299:	learn: 0.9988459	total: 25.4s	remaining: 0us
CatBoost Accuracy: 0.9920541178997101
CatBoost F1: 0.9584269662921349
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      8404
           1       0.98      0.94      0.96       909

    accuracy                           0.99      9313
   macro avg       0.99      0.97      0.98      9313
weighted avg       0.99      0.99      0.99      9313




# Focal Loss


In [10]:

class FocalLoss(nn.Module):

    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):

        ce_loss = F.cross_entropy(
            inputs,
            targets,
            reduction='none'
        )

        pt = torch.exp(-ce_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()



# GAT Model


In [11]:

class GATModel(nn.Module):

    def __init__(self, in_channels, hidden_channels, out_channels):

        super().__init__()

        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=4
        )

        self.conv2 = GATConv(
            hidden_channels * 4,
            hidden_channels
        )

        self.lin = nn.Linear(
            hidden_channels,
            out_channels
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        embeddings = self.conv2(x, edge_index)

        x = F.elu(embeddings)

        out = self.lin(x)

        return out, embeddings



# Train GAT


In [12]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GATModel(
    in_channels=data.num_node_features,
    hidden_channels=128,
    out_channels=2
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

criterion = FocalLoss()

def train():

    model.train()

    optimizer.zero_grad()

    out, _ = model(data.x, data.edge_index)

    loss = criterion(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

    return loss.item()

for epoch in range(1, 101):

    loss = train()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss:.4f}")


Epoch 10 | Loss: 0.0450
Epoch 20 | Loss: 0.0370
Epoch 30 | Loss: 0.0316
Epoch 40 | Loss: 0.0290
Epoch 50 | Loss: 0.0275
Epoch 60 | Loss: 0.0263
Epoch 70 | Loss: 0.0253
Epoch 80 | Loss: 0.0246
Epoch 90 | Loss: 0.0239
Epoch 100 | Loss: 0.0235



# Evaluate GAT


In [13]:

model.eval()

with torch.no_grad():

    out, embeddings = model(
        data.x,
        data.edge_index
    )

    preds = out.argmax(dim=1)

test_preds = preds[data.test_mask].cpu().numpy()
test_labels = data.y[data.test_mask].cpu().numpy()

print("GAT Accuracy:", accuracy_score(test_labels, test_preds))
print("GAT F1:", f1_score(test_labels, test_preds))

print(classification_report(test_labels, test_preds))


GAT Accuracy: 0.9647804144743907
GAT F1: 0.8059171597633136
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      8404
           1       0.87      0.75      0.81       909

    accuracy                           0.96      9313
   macro avg       0.92      0.87      0.89      9313
weighted avg       0.96      0.96      0.96      9313




# Hybrid GAT + CatBoost


In [14]:

embeddings_np = embeddings.cpu().numpy()

hybrid_cat = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='F1',
    class_weights=[1,5],
    verbose=50
)

hybrid_cat.fit(
    embeddings_np[train_idx],
    y_np[train_idx]
)

hybrid_preds = hybrid_cat.predict(
    embeddings_np[test_idx]
)

print("Hybrid Accuracy:", accuracy_score(test_labels, hybrid_preds))
print("Hybrid F1:", f1_score(test_labels, hybrid_preds))

print(classification_report(test_labels, hybrid_preds))


0:	learn: 0.8545491	total: 101ms	remaining: 30.2s
50:	learn: 0.9125857	total: 3.79s	remaining: 18.5s
100:	learn: 0.9405066	total: 7.43s	remaining: 14.6s
150:	learn: 0.9584673	total: 11.1s	remaining: 10.9s
200:	learn: 0.9715986	total: 14.7s	remaining: 7.23s
250:	learn: 0.9809571	total: 18.3s	remaining: 3.58s
299:	learn: 0.9883530	total: 22s	remaining: 0us
Hybrid Accuracy: 0.8223987973800064
Hybrid F1: 0.0912087912087912
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      8404
           1       0.09      0.09      0.09       909

    accuracy                           0.82      9313
   macro avg       0.50      0.50      0.50      9313
weighted avg       0.82      0.82      0.82      9313




# Save All Models


In [15]:

os.makedirs("saved_models", exist_ok=True)

# Save RF
import joblib
joblib.dump(rf_model, "saved_models/random_forest.pkl")

# Save CatBoost
cat_model.save_model("saved_models/catboost_model.cbm")

# Save GAT
torch.save(
    model.state_dict(),
    "saved_models/gat_model.pth"
)

# Save Hybrid
hybrid_cat.save_model(
    "saved_models/hybrid_gat_catboost.cbm"
)

print("Models saved successfully.")


Models saved successfully.



# Create ZIP File


In [16]:

zip_path = "elliptic_models.zip"

with zipfile.ZipFile(zip_path, "w") as zipf:

    for root, dirs, files in os.walk("saved_models"):

        for file in files:

            file_path = os.path.join(root, file)

            zipf.write(
                file_path,
                arcname=file
            )

print("ZIP created:", zip_path)


ZIP created: elliptic_models.zip



# Download Models

After execution:

- Open the right sidebar in Kaggle
- Go to Output
- Download:
    elliptic_models.zip
